In [1]:
import os
import pandas as pd

import requests
import googlemaps
import json

from geopy.distance import distance
from geopy.distance import geodesic

from dotenv import load_dotenv

from tqdm import tqdm

In [2]:
# Load environment variables
load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [3]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [4]:
centers_cache = {}
bad_locations = {}

In [ ]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

In [5]:
# Google Maps API handler
def getCenterCoords(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}", components={"country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            
            return [latitude, longitude]
        else:
            print(f"[WARNING] Could not find location {location} through google maps")
            return [None, None]
    except Exception as error:
        print(f"[ERROR] Error finding locations through google maps, {error}")
        return [None, None]

In [6]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}", components={"country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            coords = [latitude, longitude]

            # Get the city from the address components
            address_components = geocode_result[0]['address_components']
            city = None
            state = None
            for component in address_components:
                if 'locality' in component['types']:
                    city = component['long_name']
                elif 'administrative_area_level_1' in component['types']:
                    state = component['long_name']
            
            location_type = geocode_result[0]["geometry"]["location_type"]
            if location_type == "APPROXIMATE":
                bad_locations[location] = "Approximate location"
                return [None, None], None
            
            if city:
                if city in centers_cache:
                    if centers_cache[city] == coords:
                        bad_locations[location] = "Same coordinates as city"
                        return [None, None], None
                else:
                    city_coords = getCenterCoords(city)
                    centers_cache[city] = city_coords
                    if city_coords == coords:
                        bad_locations[location] = "Same coordinates as city"
                        return [None, None], None
            
            if state:
                if state in centers_cache:
                    if centers_cache[state] == coords:
                        bad_locations[location] = "Same coordinates as state"
                        return [None, None], None
                else:
                    state_coords = getCenterCoords(state)
                    centers_cache[state] = state_coords
                    if state_coords == coords:
                        bad_locations[location] = "Same coordinates as state"
                        return [None, None], None
            
            return coords, city
        else:
            print(f"[WARNING] Could not find location {location} through google maps")
            return [None, None], None
    except Exception as error:
        print(f"[ERROR] Error finding locations through google maps, {error}")
        return [None, None], None

In [7]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv()
mongo_uri = os.getenv("MONGO_URI_NAACP")
mongo_db_name = os.getenv("MONGO_DB_NAME_NAACP")
mongo_client = MongoClient(mongo_uri)

# Set up MongoDB connection
db_original = mongo_client[mongo_db_name]
locations_collection = db_original["locations_data"]
collection_name = "articles_data"
articles_collection = db_original[collection_name]


In [8]:
all_locations = locations_collection.find()
all_locations = list(all_locations)

In [9]:
# Set up MongoDB connection
new_db = mongo_client["locations_test"]
new_locations_collection = new_db["locations_data_3"]

In [10]:
reduced_locations = all_locations
for location in tqdm(reduced_locations):
    try:
        name = location["value"]
        coords = location["coordinates"]
        coords = [coords[1], coords[0]]

        general_coords, new_city = callGoogleMapsAPI(name)
        
        if coords[0] is not None and general_coords[0] is not None:
            distance_km = distance(coords, general_coords).km
        else:
            distance_km = None

        loc = {
            "value": name,
            "old_city": location["city"],
            "city": new_city,
            "old_coordinates": coords,
            "coordinates": general_coords,
            "distance_km": distance_km,
            "all_locations": [name]
        }

        same_loc_doc = new_locations_collection.find_one({"coordinates": coords})
        if same_loc_doc:
            same_loc_doc["all_locations"].append(name)
            new_locations_collection.update_one({"coordinates": coords}, {"$set": {"all_locations": same_loc_doc["all_locations"]}})
        else:
            new_locations_collection.insert_one(loc)

    except Exception as error:
        print(f"[ERROR] Error processing location {location['value']}, {error}")
        continue



 29%|██▊       | 1328/4646 [06:11<19:53,  2.78it/s]

[WARNING] Could not find location laredo colombia solidarity international bridge through google maps


 33%|███▎      | 1516/4646 [07:11<14:50,  3.51it/s]

[WARNING] Could not find location blue ledge through google maps


 95%|█████████▌| 4434/4646 [19:16<00:51,  4.08it/s]

[ERROR] Error processing location national naacp convention seaport 3, 'coordinates'


 98%|█████████▊| 4567/4646 [19:48<00:12,  6.56it/s]

[ERROR] Error processing location kraft center for community health, 'coordinates'


100%|██████████| 4646/4646 [20:05<00:00,  3.85it/s]


In [15]:
new_locations = new_locations_collection.find()


[{'_id': ObjectId('66eda4aa923d548143a8aeca'), 'value': 'mgh center for disaster medicine', 'old_city': 'Boston', 'city': 'Boston', 'old_coordinates': [42.36256789999999, -71.068767], 'coordinates': [42.36256789999999, -71.068767], 'distance_km': 0.0, 'all_locations': ['mgh center for disaster medicine', 'massachusetts general hospital', 'mass general', 'mass general hospital', 'mgh', 'mass general division of infectious diseases', 'boston medical center mgh', 'massachusetts general hospital center for global health', 'mass general hospital mongan institute', 'mgh center for global health', 'massachusetts general hospital community access recruitment and engagement research center', 'massachusetts general hospital center for disaster medicine', 'mass general hospital center for gun violence prevention', 'massachusetts general hospital comprehensive sickle cell disease treatment center']}, {'_id': ObjectId('66eda4aa923d548143a8aecb'), 'value': 'department of public health', 'old_city': 

In [22]:
new_locations_df = pd.DataFrame(new_locations)
new_locations_df["coordinates"] = new_locations_df["coordinates"].apply(lambda x: x if x != [None, None] else None)
filtered_df = new_locations_df[new_locations_df["coordinates"].notnull()]
print(len(filtered_df))

2102


In [23]:
new_locations_collection = new_db["locations_data_4"]
new_locations_collection.insert_many(filtered_df.to_dict("records"))

InsertManyResult([ObjectId('66eda4aa923d548143a8aeca'), ObjectId('66eda4aa923d548143a8aecb'), ObjectId('66eda4ac923d548143a8aed3'), ObjectId('66eda4ac923d548143a8aed5'), ObjectId('66eda4ac923d548143a8aed8'), ObjectId('66eda4ad923d548143a8aed9'), ObjectId('66eda4ad923d548143a8aedc'), ObjectId('66eda4ad923d548143a8aedd'), ObjectId('66eda4ae923d548143a8aede'), ObjectId('66eda4ae923d548143a8aee0'), ObjectId('66eda4ae923d548143a8aee1'), ObjectId('66eda4af923d548143a8aee2'), ObjectId('66eda4af923d548143a8aee3'), ObjectId('66eda4af923d548143a8aee4'), ObjectId('66eda4b0923d548143a8aee8'), ObjectId('66eda4b0923d548143a8aee9'), ObjectId('66eda4b0923d548143a8aeea'), ObjectId('66eda4b1923d548143a8aeec'), ObjectId('66eda4b1923d548143a8aeed'), ObjectId('66eda4b1923d548143a8aeef'), ObjectId('66eda4b1923d548143a8aef0'), ObjectId('66eda4b2923d548143a8aef4'), ObjectId('66eda4b2923d548143a8aef5'), ObjectId('66eda4b3923d548143a8aef6'), ObjectId('66eda4b3923d548143a8aef7'), ObjectId('66eda4b3923d548143a8ae

In [25]:
updated_locations = new_locations_collection.find()
updated_locations = list(updated_locations)

In [28]:
for entry in tqdm(updated_locations):
    locations = entry["all_locations"]
    if len(locations) > 1:
        possible_locations = []
        for location in locations:
            if len(location) > 3:
                possible_locations.append(location)

        locations = possible_locations
        
        if len(locations) < 2:
            entry["location"] = locations[0]
            continue

        sorted_locations = sorted(locations, key=len)

        def find_substring(sorted_list):
            for i in range(len(sorted_list)):
                substring = sorted_list[i]

                # Check if this substring is in at least some of the others
                count = sum(1 for other in sorted_list if substring in other and other != substring)
                if count > 0:
                    return substring
            return None

        # Get the result
        result = find_substring(sorted_locations)
        if result:
            entry["location"] = result
        else:
            if len(sorted_locations[0]) > 1:
                if len(sorted_locations[0].split(" ")) > 1:
                    entry["location"] = sorted_locations[0]
                else:
                    entry["location"] = sorted_locations[1]
            else:
                entry["location"] = sorted_locations[0]
    else:
        entry["location"] = locations[0]
    
    new_locations_collection.update_one({"coordinates": entry["coordinates"]}, {"$set": {"value": entry["location"]}})

100%|██████████| 2102/2102 [02:15<00:00, 15.52it/s]


In [ ]:
new_articles = new_db["articles_data"]
all_articles = articles_collection.find()
all_articles = list(all_articles)
for article in tqdm(all_articles):
    location_coordinates = article["locations"]

In [2]:
print(bad_locations.keys())

NameError: name 'bad_locations' is not defined

In [1]:
for location in bad_locations:
    if bad_locations[location] != "Approximate location":
        print(location, bad_locations[location])


NameError: name 'bad_locations' is not defined